In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans Mono'
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from data import load_data

df = load_data()

X_str = df.iloc[:, 3:].values
X_oe = OrdinalEncoder().fit_transform(X_str)
y = df['Geographic region'].values
y_enc = LabelEncoder().fit_transform(y)
min_categories = df.iloc[:, 3:].nunique().values

splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

import itertools

class GaussianMixturePP(GaussianMixture):
    def fit(self, X, y):
        super().fit(X, y)
        labels = np.unique(y)
        y_recon = super().predict(X)
        perms = list(
            labels[list(perm)] for perm in itertools.permutations(range(self.n_components))
        )
        scores = np.empty((len(perms), ))
        for i, perm in enumerate(perms):
            y_perm = np.vectorize(perm.__getitem__)(y_recon)
            scores[i] = np.mean(y_perm == y)
        self.optimal_perm = perms[np.argmax(scores)]
        return self
    
    def predict(self, X, y=None):
        return np.vectorize(self.optimal_perm.__getitem__)(super().predict(X))


moo = make_pipeline(
    OneHotEncoder(
        handle_unknown='ignore',
        sparse=False
    ),
    PCA(
        # n_components=30,
        n_components=5,
        random_state=42
    ),
    # UMAP(
    #     n_components=3,
    #     random_state=42
    # ),
    # OrdinalEncoder(
    #     handle_unknown='use_encoded_value',
    #     unknown_value=-1
    # ),
    # SGDClassifier(
    #     loss='log',
    #     random_state=42
    # ),
    # SVC(
    #     gamma=1e-2,
    #     random_state=42
    # )
    # KNeighborsClassifier(
    #     n_neighbors=5,
    # )
    # CategoricalNB(
    #     min_categories=min_categories
    # )
    GaussianMixturePP(
        n_components=3,
        random_state=42
    )
    # DecisionTreeClassifier(
    #     max_depth=4,
    #     # min_samples_split=10,
    #     # min_samples_leaf=5,
    #     criterion='entropy',
    #     random_state=42
    # )
    # LogisticRegression(
    #     penalty='l1',
    #     solver='liblinear',
    #     max_iter=1000,
    #     random_state=42
    # )
    
)

for train_idxs, test_idxs in splitter.split(X_oe, y):
    moo.fit(X_oe[train_idxs], y[train_idxs])
    print(accuracy_score(y[train_idxs], moo.predict(X_oe[train_idxs])))
    print(accuracy_score(y[test_idxs], moo.predict(X_oe[test_idxs])))
    break

0.8386908240794857
0.8271028037383178


In [19]:
models = [
    ('pca+l2', make_pipeline(
        OneHotEncoder(
            handle_unknown='ignore',
            sparse=False,
        ),
        PCA(
            n_components=30,
            random_state=42
        ),
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )),
    ('pca+l1', make_pipeline(
        OneHotEncoder(
            handle_unknown='ignore',
            sparse=False,
        ),
        PCA(
            n_components=30,
            random_state=42
        ),
        LogisticRegression(
            solver='liblinear',
            penalty='l1',
            max_iter=1000,
            random_state=42
        )
    )),
    ('pca+DT', make_pipeline(
        OneHotEncoder(
            handle_unknown='ignore',
            sparse=False,
        ),
        PCA(
            n_components=30,
            random_state=42
        ),
        DecisionTreeClassifier(
            max_depth=4,
            criterion='entropy',
            random_state=42
        )
    )),
    ('pca+GM', make_pipeline(
        OneHotEncoder(
            handle_unknown='ignore',
            sparse=False,
        ),
        PCA(
            n_components=5,
            random_state=42
        ),
        GaussianMixturePP(
            n_components=3,
            random_state=42
        )
    )),
    ('CatNB', make_pipeline(
        CategoricalNB(
            min_categories=min_categories
        )
    ))
]

columns = ['SNU-ID', 'true_label']
for model_name, _ in models:
    columns += [f'{model_name}_pred_label', f'{model_name}_misclassified']
columns += [
    'NEA_misclassified_as_SEA_count',
    'NEA_misclassified_as_SWA_count',
    'SEA_misclassified_as_NEA_count',
    'SEA_misclassified_as_SWA_count',
    'SWA_misclassified_as_NEA_count',
    'SWA_misclassified_as_SEA_count'
]
df_out = pd.DataFrame(
    data=np.empty((df.shape[0], len(columns))),
    columns=columns
)
df_out['SNU-ID'] = df['SNU-ID']
df_out['true_label'] = df['Geographic region']

for model_name, model in models:
    y_oof = np.empty_like(y)
    for train_idxs, test_idxs in splitter.split(X_oe, y):
        model.fit(X_oe[train_idxs], y[train_idxs])
        y_oof[test_idxs] = model.predict(X_oe[test_idxs])
    df_out[f'{model_name}_pred_label'] = y_oof
    df_out[f'{model_name}_misclassified'] = (y_oof != y)
for label_true, label_pred in itertools.permutations(['NEA', 'SEA', 'SWA'], 2):
    df_out[f'{label_true}_misclassified_as_{label_pred}_count'] \
        = np.sum((df_out['true_label'] == label_true) & (df_out[[f'{model_name}_pred_label' for model_name, _ in models]] == label_pred).T, axis=0)
df_out.to_csv('../output/230516_region_classification.csv', index=False)

In [67]:
def ent(s):
    s = s.value_counts(normalize=True).values
    return -s @ np.log(s)
df.iloc[:, 3:].apply(ent).sort_values().values

array([0.        , 0.0040523 , 0.0458199 , 0.09534777, 0.09698045,
       0.10966289, 0.11841149, 0.171541  , 0.20123397, 0.23485148,
       0.25216183, 0.28892599, 0.32829718, 0.34809837, 0.37418613,
       0.39744728, 0.39966449, 0.46105769, 0.49117651, 0.4921611 ,
       0.49400588, 0.49753235, 0.51091577, 0.51247079, 0.51429283,
       0.52010629, 0.5323931 , 0.54891961, 0.56024102, 0.56099399,
       0.57372756, 0.57864134, 0.59983164, 0.62942735, 0.6559563 ,
       0.67027129, 0.67251486, 0.67445188, 0.67567295, 0.67599233,
       0.69144188, 0.69148667, 0.69180424, 0.70942661, 0.71011862,
       0.71136581, 0.72201709, 0.72782166, 0.73242968, 0.74216269,
       0.74470398, 0.75639821, 0.77344595, 0.78738506, 0.80693994,
       0.80736608, 0.80805086, 0.81005649, 0.81517728, 0.82162237,
       0.82219756, 0.82450565, 0.83159222, 0.83561571, 0.84395492,
       0.84409767, 0.84975721, 0.85198026, 0.85824129, 0.85924628,
       0.86053048, 0.86643527, 0.87017018, 0.87431037, 0.87585

In [1]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder
# from sklearn.model_selection import StratifiedKFold
# from data import load_data

# plt.rcParams['font.family'] = 'DejaVu Sans Mono'

# df = load_data()
# genes = [el for el in df.columns if el.startswith('rs')]

# df_coded = df.copy()[genes]
# mask_nontrivial = []
# for gene in genes:
#     col = df[gene]
#     cnt_a, cnt_c, cnt_g, cnt_t = [col.str.count(char) for char in 'ACGT']
#     df_coded[gene] = (cnt_a*27 + cnt_c*9 + cnt_g*3 + cnt_t).astype(np.float32)
#     mask_nontrivial += [cnt.nunique() for cnt in [cnt_a, cnt_c, cnt_g, cnt_t]]
# mask_nontrivial = np.array(mask_nontrivial) > 1

# np.seterr(invalid='ignore')

# class DFTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self, goal, columns=genes):
#         super().__init__()
#         if goal == 'cat':
#             self.inner_tfmer = OrdinalEncoder().fit(df_coded.loc[:, columns])
#         elif goal == 'oh':
#             self.inner_tfmer = OneHotEncoder(sparse=False).fit(df_coded.loc[:, columns])
#         elif goal == 'num':
#             self.inner_tfmer = FunctionTransformer().fit(df_coded)
#         self.goal = goal
#         if columns is None:
#             columns = np.full((len(genes), ), True)
#         self.columns = columns
    
#     def fit(self, X, y=None):
#         return self

#     def transform(self, X, y=None):
#         return self.inner_tfmer.transform(self._pretransform(X))

#     def _pretransform(self, X):
#         if not isinstance(self.inner_tfmer, FunctionTransformer):
#             return X.values
#         else:
#             arr = np.where(X.values != 0, X.values, np.nan)
#             result_a, arr = np.divmod(arr, 27)
#             result_c, arr = np.divmod(arr, 9)
#             result_g, result_t = np.divmod(arr, 3)
#             result = np.stack([result_a, result_c, result_g, result_t], axis=-1).reshape(arr.shape[0], -1)
#             return result[:, mask_nontrivial]
            
# def cv_splitter(X, y, n_splits, shuffle=True, random_state=None):
#     for train_idxs, test_idxs in StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state).split(X, y):
#         yield X.iloc[train_idxs], X.iloc[test_idxs], y[train_idxs], y[test_idxs]